# Week 2 · Exercises: ML from scratch in NumPy

- Each exercise is one piece of linear regression, logistic regression, or kNN, built **from scratch** with NumPy, and then checked against scikit-learn.
- Run the first cell to load the checker. Write on paper first: these are the exam's "write the algorithm by hand" questions.
- **Watch the shapes** ([notebook 03 ↗](03_shapes_and_slicing.ipynb)): every function gets `(n, d)` arrays and returns the shape stated in the description.
- The tests use randomly generated data with different sizes every run, so they can't pass by luck (the lesson from `chunk` and `common_keys`, see [00 §A ↗](00_review_and_recap.ipynb), points 2 and 6d).

The solutions are in `solutions/week2_solutions.py`.

In [1]:
import numpy as np

_rng = np.random.default_rng()

def _data(n=None, d=None, noise=0.1):
    n = n or int(_rng.integers(30, 80)); d = d or int(_rng.integers(2, 6))
    X = _rng.normal(size=(n, d)); w = _rng.normal(size=d)
    return X, X @ w + 1.5 + noise * _rng.normal(size=n)

def _t1():
    X, _ = _data()
    A = add_bias(X)
    return A.shape == (X.shape[0], X.shape[1] + 1) and np.allclose(A[:, 0], 1) and np.allclose(A[:, 1:], X)

def _t2():
    X, y = _data()
    w = ols_fit(X, y)
    ref = np.linalg.lstsq(np.column_stack([np.ones(len(X)), X]), y, rcond=None)[0]
    return w.shape == (X.shape[1] + 1,) and np.allclose(w, ref)

def _t3():
    y = _rng.normal(size=50); e = 0.3 * _rng.normal(size=50)
    from sklearn.metrics import r2_score as R
    return np.isclose(r2(y, y + e), R(y, y + e)) and np.isclose(r2(y, y), 1.0) and np.isclose(r2(y, np.full(50, y.mean())), 0.0)

def _t4():
    from sklearn.linear_model import Ridge
    X, y = _data(); lam = float(_rng.uniform(0.1, 10))
    w = ridge_fit(X, y, lam)
    ref = Ridge(alpha=lam, fit_intercept=False).fit(X, y).coef_
    return w.shape == (X.shape[1],) and np.allclose(w, ref, atol=1e-6) and np.allclose(ridge_fit(X, y, 0.0), np.linalg.lstsq(X, y, rcond=None)[0])

def _t5():
    X, y = _data(noise=0.0)
    w = gd_linear(add_bias(X), y, lr=0.1, steps=3000)
    return w.shape == (X.shape[1] + 1,) and np.allclose(w, ols_fit(X, y), atol=1e-3)

def _t6():
    z = np.array([-3.0, 0.0, 2.0])
    return np.allclose(sigmoid(z), 1 / (1 + np.exp(-z))) and np.isclose(sigmoid(0.0), 0.5) and sigmoid(np.zeros((2, 3))).shape == (2, 3)

def _t7():
    y = np.array([1, 0, 1, 1]); p = np.array([0.9, 0.2, 0.6, 0.99])
    if bce(y, p) is None:
        raise TypeError("'NoneType' returned")
    ref = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    safe = np.isfinite(bce(np.array([1, 0]), np.array([1.0, 0.0])))      # p exactly 0 or 1 must not give inf/nan
    return np.isclose(bce(y, p), ref) and safe

def _t8():
    n = int(_rng.integers(80, 150)); X = _rng.normal(size=(n, 2))
    y = (X @ np.array([2.0, -1.0]) + 0.3 > 0).astype(int)
    A = add_bias(X); lr = float(_rng.choice([0.1, 0.5])); steps = int(_rng.integers(200, 800))
    w = logistic_gd(A, y, lr=lr, steps=steps)
    ref = np.zeros(3)                                   # the exact update from the description
    for _ in range(steps):
        ref = ref - lr * (1 / n) * A.T @ (1 / (1 + np.exp(-(A @ ref))) - y)
    acc = ((1 / (1 + np.exp(-(A @ w))) >= 0.5).astype(int) == y).mean()
    return w.shape == (3,) and np.allclose(w, ref) and acc >= 0.9

def _t9():
    y = _rng.integers(0, 2, 60); yh = _rng.integers(0, 2, 60)
    from sklearn.metrics import confusion_matrix
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return tuple(int(v) for v in confusion_counts(y, yh)) == (tp, fp, fn, tn)

def _t10():
    from sklearn.metrics import precision_score, recall_score, f1_score
    y = _rng.integers(0, 2, 80); yh = _rng.integers(0, 2, 80)
    p, r, f = precision_recall_f1(y, yh)
    ok = np.allclose([p, r, f], [precision_score(y, yh), recall_score(y, yh), f1_score(y, yh)])
    zero = precision_recall_f1(np.array([1, 1]), np.array([0, 0]))      # no positive predictions → 0, not a crash
    return ok and tuple(float(v) for v in zero) == (0.0, 0.0, 0.0)

def _t11():
    from sklearn.neighbors import KNeighborsClassifier
    Xtr = _rng.normal(size=(60, 2)) + np.repeat([[0, 0], [3, 3]], 30, axis=0); ytr = np.repeat([0, 1], 30)
    Xte = _rng.uniform(-1, 4, size=(25, 2)); k = int(_rng.choice([1, 3, 5, 7]))
    ref = KNeighborsClassifier(n_neighbors=k).fit(Xtr, ytr).predict(Xte)
    out = knn_predict(Xtr, ytr, Xte, k)
    return out.shape == (25,) and (out == ref).mean() >= 0.96

def _t12():
    Xtr = _rng.normal(5, 2, size=(40, 3)); Xte = _rng.normal(5, 2, size=(10, 3))
    Ztr, Zte = scale_train_test(Xtr, Xte)
    mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)
    return np.allclose(Ztr, (Xtr - mu) / sd) and np.allclose(Zte, (Xte - mu) / sd) and not np.allclose(Zte.mean(axis=0), 0)

TESTS = {1: _t1, 2: _t2, 3: _t3, 4: _t4, 5: _t5, 6: _t6, 7: _t7, 8: _t8, 9: _t9, 10: _t10, 11: _t11, 12: _t12}


def check(n):
    """Run the hidden tests for exercise n and say whether they pass."""
    try:
        ok = bool(TESTS[n]())
        print(f"Exercise {n}: " + ("PASS" if ok else "not right yet. A test case gives the wrong answer (re-read the description and try the edge cases)"))
    except NameError as e:
        print(f"Exercise {n}: run the cell with your code first ({e})")
    except TypeError as e:
        if "NoneType" in str(e):
            print(f"Exercise {n}: something returned None. Not written yet, or a missing `return`?")
        elif "takes no arguments" in str(e):
            print(f"Exercise {n}: your class has no __init__ yet, or its __init__ doesn't take these inputs ({e})")
        else:
            print(f"Exercise {n}: your code raised TypeError: {e}")
    except AttributeError as e:
        if "'NoneType'" in str(e):
            print(f"Exercise {n}: something returned None. Not written yet, or a missing `return`?")
        else:
            print(f"Exercise {n}: a method or attribute is missing: {e}. Check the spelling, `self.`, and super().__init__()")
    except Exception as e:
        print(f"Exercise {n}: your code raised {type(e).__name__}: {e}")

## 1. Add the intercept column
Return `X` with a column of 1s added at the **front**. Input shape `(n, d)`, output shape `(n, d + 1)`. Hint: `np.column_stack` ([notebook 03 §4 ↗](03_shapes_and_slicing.ipynb)).

In [15]:
def add_bias(X):
    return np.column_stack([np.ones(X.shape[0]),X])

In [16]:
check(1)

Exercise 1: PASS


## 2. Ordinary least squares
Fit linear regression **with an intercept** by solving the normal equation $X^\top X\, w = X^\top y$, using your `add_bias`. Return `w` of shape `(d + 1,)`: the intercept first, then one weight per feature. Use `np.linalg.solve`, not `inv`.

In [19]:
def ols_fit(X, y):
    X_new = add_bias(X)
    return np.linalg.solve(X_new.T@X_new,X_new.T@y)

In [20]:
check(2)

Exercise 2: PASS


## 3. $R^2$
Return $R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}$.

In [31]:
def r2(y, y_hat):
    return 1 - np.sum((y - y_hat)**2)/np.sum((y - np.mean(y))**2)

In [32]:
check(3)

Exercise 3: PASS


## 4. Ridge regression (no intercept)
Return $\hat w = (X^\top X + \lambda I)^{-1} X^\top y$ for data **without** a bias column, shape `(d,)`. `np.eye(d)` makes $I$. Check that `lam = 0` gives ordinary least squares.

In [46]:
def ridge_fit(X, y, lam):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)

In [47]:
check(4)

Exercise 4: PASS


### C.F)
### Ridge and the intercept: `I[0, 0] = 0`

The intercept $b$ should **not** be penalized. With a ones column and the full identity, $\lambda$ also shrinks $b$, which pulls all predictions toward 0. Fix: zero the bias entry of $I$.

```python
X_new = add_bias(X)                  # ones column first
I = np.eye(X_new.shape[1])
I[0, 0] = 0                          # row 0, column 0 → no penalty on b
w = np.linalg.solve(X_new.T @ X_new + lam * I, X_new.T @ y)
```

More common alternative (ESL, sklearn): center $X$ and $y$, fit ridge without intercept, then $b = \bar y - \bar x^\top w$. Same result.

If the task says no bias column, skip both: `np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)`.

## 5. Gradient descent for linear regression
`X` already has its bias column. Start from `w = 0`, then repeat `steps` times: $w \leftarrow w - \eta \cdot \frac{2}{n} X^\top (Xw - y)$. Return `w`. It should end up equal to the OLS answer.

In [ ]:
def gd_linear(X, y, lr, steps):
    n , d = X.shape
    w = np.zeros(d)
    for _ in range(steps):
        grad = 2 / n*(X.T@X@w - X.T@y)
        w -= lr*grad
    return w

# had some problem solving this.
# my first answer looked like the below:
#def gd_linear(X, y, lr, steps):
    w = 0
    for _ in range(steps):
        w -= lr*2*np.mean(X.T@X*w - X.T@y)
    return w
# 1. w = 0 makes the weights a single number. It should be a vector with one weight per feature: w = np.zeros(X.shape[1]). With a single number, every feature shares the same weight.
# 2. np.mean averages over the wrong thing. X.T @ X @ w - X.T @ y is a vector with one entry per feature. np.mean averaged those entries into one number, so the gradient collapsed to a scalar and every weight got the same update. The $\frac{1}{n}$ in the formula is supposed to divide by the number of samples: 2 / n * X.T @ (X @ w - y).
# 3. * instead of @ in X.T @ X * w. Once w is a vector, * multiplies element by element (each column of $X^\top X$ by one weight) and gives a $(d \times d)$ matrix instead of the $(d)$ vector $X^\top X w$. You need @ for matrix-vector multiplication. This one was hidden only because w was a number.

In [73]:
check(5)

Exercise 5: PASS


## 6. Sigmoid
Return $\sigma(z) = \frac{1}{1 + e^{-z}}$ for an array of any shape (or a single number).

In [74]:
def sigmoid(z):
    return 1/(1+ np.exp(-z))

In [75]:
check(6)

Exercise 6: PASS


## 7. Binary cross-entropy
Return the mean of $-[y \log p + (1 - y) \log(1 - p)]$. **First clip** `p` into `[1e-12, 1 - 1e-12]` with `np.clip`, so that a prediction of exactly 0 or 1 doesn't give `log(0) = -inf`.

In [92]:
def bce(y, p):
    new_p = np.clip(p,1e-12,1 - 1e-12)
    return np.mean(-(y*np.log(new_p) + (1-y)*np.log(1-new_p)))

# two mistakes i made
# 1. The clipped result was thrown away. np.clip doesn’t change p in place. It returns a new array, and your line didn’t store it anywhere, so p still contains exact 0s and 1s and np.log can still return -inf. Fix: p = np.clip(p, 1e-12, 1 - 1e-12). --> this is kind of a trivial mistake so no need to review i believe. I leave the assessment to you
# 2. @ instead of *. On two 1-D arrays, @ is the dot product: it multiplies element by element and adds everything up into one number. So y @ np.log(p) + (1 - y) @ np.log(1 - p) is already the sum of the losses, and np.mean of a single number does nothing. You get the total loss, $n$ times too big, instead of the average. With * you keep one loss per sample, and np.mean then averages them.

In [93]:
check(7)

Exercise 7: PASS


## 8. Logistic regression by gradient descent
`X` already has its bias column, and `y` holds 0s and 1s. Start from `w = 0` and repeat: $w \leftarrow w - \eta \cdot \frac{1}{n} X^\top (\sigma(Xw) - y)$. Return `w`. The test runs exactly this update itself and checks that your `w` matches, and that the model classifies its dataset with at least 90% accuracy.

In [100]:
def logistic_gd(X, y, lr, steps):
    n, d = X.shape
    w = np.zeros(d)
    for _ in range(steps):
        grad = X.T@(sigmoid(X@w)-y)/n
        w -= lr*grad
    return w

# made a mistake where i had gad outside of the for iteration. minor mistake?

In [101]:
check(8)

Exercise 8: PASS


## 9. Confusion counts
Return the tuple `(tp, fp, fn, tn)` for 0/1 arrays `y` (truth) and `y_hat` (prediction). One line each, with masks: e.g. $TP$ is `((y == 1) & (y_hat == 1)).sum()`.

In [104]:
def confusion_counts(y, y_hat):
    tp = ((y == 1) & (y_hat == 1)).sum()
    fp = ((y == 0) & (y_hat == 1)).sum()
    fn = ((y == 1) & (y_hat == 0)).sum()
    tn = ((y == 0) & (y_hat == 0)).sum()
    return (tp, fp, fn, tn)

In [105]:
check(9)

Exercise 9: PASS


## 10. Precision, recall, $F_1$
Return `(precision, recall, f1)` using your `confusion_counts`. If a denominator is 0 (e.g. the model never predicts 1), return 0.0 for that metric instead of crashing.

In [106]:
def precision_recall_f1(y, y_hat):
    tp, fp, fn, tn = confusion_counts(y, y_hat)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return (float(precision), float(recall), float(f1))

In [107]:
check(10)

Exercise 10: PASS


## 11. k-nearest neighbors
For each row of `X_test`, compute its distance to every training point, take the `k` nearest (`np.argsort`), and return the majority label (`np.bincount(...).argmax()` works for 0/1 labels). Output shape `(len(X_test),)`. Bonus: do the distances for all test points at once with broadcasting: `X_test[:, None, :] - X_train[None, :, :]` has shape `(m, n, d)`.

**The broadcasting trick, drawn.** Add a new axis to each array so that every test point meets every training point:

```text
X_test[:, None, :]   shape (m, 1, d)     m test points,  one slot for "which training point"
X_train[None, :, :]  shape (1, n, d)     one slot for "which test point",  n training points

line up from the right:   d vs d ✓    1 vs n → stretch    m vs 1 → stretch

difference           shape (m, n, d)     [i, j] = test point i minus training point j
norm(..., axis=2)    shape (m, n)        [i, j] = distance from test i to training j
argsort(axis=1)[:, :k]  shape (m, k)     for each test point, its k nearest training rows
```

Example: 2 test points and 3 training points in 2-D give a difference of shape `(2, 3, 2)` and distances of shape `(2, 3)`: one row per test point, one column per training point.

In [ ]:
def knn_predict(X_train, y_train, X_test, k):
    

In [ ]:
check(11)

## 12. Standardize without leakage
Return `(Z_train, Z_test)`: both standardized with the **training** mean and standard deviation. The test set must **not** be standardized with its own statistics. That's the whole point of the exercise.

In [ ]:
def scale_train_test(X_train, X_test):
    

In [ ]:
check(12)

---
### Run all checks

In [ ]:
for n in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
    check(n)

When all 12 pass, commit and push, then log your hours and which exercises needed hints.